In [1]:
print("String")

String


In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("uraninjo/augmented-alzheimer-mri-dataset-v2")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset-v2


In [3]:
import os
import random
import time
import gc
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.applications import DenseNet121
from tensorflow.keras.applications.densenet import preprocess_input
from tensorflow.keras.layers import Dense, Flatten, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.model_selection import StratifiedKFold, train_test_split

# ==========================================
# 1. THE SCIENTIFIC LOCKS (T-TEST SYNCHRONIZATION)
# ==========================================
SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = '1'
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# ==========================================
# 2. DUAL-GPU HARDWARE SETUP (T4 x2)
# ==========================================
if 'strategy' not in locals():
    try:
        strategy = tf.distribute.MirroredStrategy()
        print(f"✅ Dual-GPU Strategy Initialized. Replicas: {strategy.num_replicas_in_sync}")
    except Exception as e:
        print(f"❌ GPU initialization failed: {e}")
        strategy = tf.distribute.get_strategy()
else:
    print(f"⚡ Strategy already active. Replicas: {strategy.num_replicas_in_sync}")

BATCH_SIZE = 32 * strategy.num_replicas_in_sync 
AUTOTUNE = tf.data.AUTOTUNE

# ==========================================
# 3. DATA LOADING (WITH T-TEST SORTING FIX)
# ==========================================
DATA_DIR = '/kaggle/input/datasets/uraninjo/augmented-alzheimer-mri-dataset-v2/data'

print("\nFetching Local File Paths...")
filepaths = []

for root, dirs, files in os.walk(DATA_DIR):
    for file in files:
        if file.lower().endswith(('.png', '.jpg', '.jpeg')):
            filepaths.append(os.path.join(root, file))

if len(filepaths) == 0:
    print(f"❌ ERROR: No images found in {DATA_DIR}.")

# 🚨 CRUCIAL FOR PAIRED T-TEST: Sorts paths so all teammates have identical folds 🚨
filepaths.sort()

labels = [os.path.basename(os.path.dirname(path)) for path in filepaths]

df = pd.DataFrame({'filepath': filepaths, 'label': labels})
class_names = sorted(df['label'].unique())
class_indices = {name: idx for idx, name in enumerate(class_names)}
df['label_idx'] = df['label'].map(class_indices)

print(f"✅ Successfully pooled and sorted {len(df)} images.")
print(f"Classes found: {class_indices}")

# ==========================================
# 4. TF.DATA PIPELINE FUNCTIONS 
# ==========================================
def decode_image(filepath, label):
    """Reads, decodes, resizes, and pre-processes specifically for DenseNet121."""
    image_bytes = tf.io.read_file(filepath)
    image = tf.image.decode_jpeg(image_bytes, channels=3)
    image = tf.image.resize(image, [224, 224])
    
    # DenseNet requires its own specific math to center the pixels
    image = preprocess_input(image)
    
    label = tf.one_hot(label, depth=len(class_names))
    return image, label

def build_dataset(dataframe, is_training=False):
    dataset = tf.data.Dataset.from_tensor_slices((dataframe['filepath'].values, dataframe['label_idx'].values))
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    
    if is_training:
        dataset = dataset.shuffle(buffer_size=2048, seed=SEED)
        
    dataset = dataset.batch(BATCH_SIZE)
    dataset = dataset.prefetch(AUTOTUNE) 
    return dataset

# ==========================================
# 5. MODEL ARCHITECTURE BUILDER (DenseNet121)
# ==========================================
def build_densenet_model():
    tf.keras.backend.clear_session()
    
    with strategy.scope():
        # Swapped to DenseNet121
        base_model = DenseNet121(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        base_model.trainable = False 
        
        x = base_model.output
        x = GlobalAveragePooling2D()(x) 
        x = Dense(256, activation='relu')(x)
        x = Dropout(0.5)(x)
        outputs = Dense(len(class_names), activation='softmax')(x) 
        
        model = Model(inputs=base_model.input, outputs=outputs)
        
        model.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )
    return model

# ==========================================
# 6. THE 5-FOLD CROSS VALIDATION LOOP
# ==========================================
K_FOLDS = 5 
EPOCHS = 50

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
fold_accuracies = []

print("\n🚀 STARTING 5-FOLD CROSS-VALIDATION (DenseNet121 on T4 x2 GPUs)...")
start_time = time.time()

for fold, (train_val_idx, test_idx) in enumerate(skf.split(df['filepath'], df['label_idx'])):
    fold_num = fold + 1
    print(f"\n{'='*50}\n🧪 STARTING FOLD {fold_num} / {K_FOLDS}\n{'='*50}")
    
    test_df = df.iloc[test_idx]
    train_val_df = df.iloc[train_val_idx]
    
    train_df, val_df = train_test_split(
        train_val_df, 
        test_size=0.1, 
        stratify=train_val_df['label_idx'], 
        random_state=SEED
    )
    
    train_ds = build_dataset(train_df, is_training=True)
    val_ds = build_dataset(val_df, is_training=False)
    test_ds = build_dataset(test_df, is_training=False)
    
    model = build_densenet_model()
    
    # Save paths specifically named for DenseNet
    checkpoint_path = f'/kaggle/working/best_densenet121_fold_{fold_num}.keras'
    checkpoint = ModelCheckpoint(checkpoint_path, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1)
    early_stopping = EarlyStopping(monitor='val_accuracy', patience=10, restore_best_weights=True, verbose=1)
    
    print("\n⏳ Training Model...")
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=[checkpoint, early_stopping],
        verbose=1
    )
    
    print("\n🔍 Evaluating on unseen Pure Test Set...")
    model.load_weights(checkpoint_path)
    test_loss, test_acc = model.evaluate(test_ds, verbose=0)
    fold_accuracies.append(test_acc)
    
    print(f"✅ Fold {fold_num} Final Test Accuracy: {test_acc * 100:.2f}%")

    print("🧹 Wiping GPU Memory clean for the next fold...")
    del model
    del train_ds, val_ds, test_ds
    tf.keras.backend.clear_session()
    gc.collect()

# ==========================================
# 7. SAVE FINAL RESULTS
# ==========================================
final_array = np.array(fold_accuracies)
np.save('/kaggle/working/DenseNet121_5Fold_Accuracies_DualGPU.npy', final_array)

print(f"\n🎉 ALL {K_FOLDS} FOLDS COMPLETE!")
print(f"🏆 Average Test Accuracy: {np.mean(fold_accuracies) * 100:.2f}% (+/- {np.std(fold_accuracies) * 100:.2f}%)")
print(f"⏱️ Total Experiment Time: {(time.time() - start_time) / 60:.2f} minutes")

2026-04-01 10:34:48.767057: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1775039689.158321      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1775039689.273855      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1775039690.247717      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039690.247756      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1775039690.247759      24 computation_placer.cc:177] computation placer alr

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')


I0000 00:00:1775039728.361347      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1775039728.367222      24 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


✅ Dual-GPU Strategy Initialized. Replicas: 2

Fetching Local File Paths...
✅ Successfully pooled and sorted 40384 images.
Classes found: {'MildDemented': 0, 'ModerateDemented': 1, 'NonDemented': 2, 'VeryMildDemented': 3}

🚀 STARTING 5-FOLD CROSS-VALIDATION (DenseNet121 on T4 x2 GPUs)...

🧪 STARTING FOLD 1 / 5
29084464/29084464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step

⏳ Training Model...
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0

I0000 00:00:1775039814.352782      69 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1775039815.052982      72 cuda_dnn.cc:529] Loaded cuDNN version 91002


455/455 ━━━━━━━━━━━━━━━━━━━━ 0s 172ms/step - accuracy: 0.3726 - loss: 1.4601INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).

Epoch 1: val_accuracy improved from -inf to 0.58743, saving model to /kaggle/working/best_densenet121_fold_1.keras
455/455 ━━━━━━━━━━━━━━━━━━━━ 117s 212ms/step - accuracy: 0.3728 - loss: 1.4596 - val_accuracy: 0.5874 - val_loss: 0.9814
Epoch 2/50
455/455 ━━━━━━━━━━━━━━━━━━━━ 0s 164ms/step - accuracy: 0.5521 - loss: 1.0067
Epoch 2: val_accuracy improved from 0.58743 to 0.62736, saving model to /kaggle/working/best_densenet121_fold_1.keras
455/455 ━━━━━━━━━━━━━━━━━━━━ 87s 184ms/step - accuracy: 0.5521 - loss: 1.0066 - val_accuracy: 0.6274 - val_loss: 0.8754
Epoch 3/50
455/455 ━━━━━━━━━━━━━━━━━━━━ 0s 161ms/step - accuracy: 0.5855 - loss: 